# Optuna

In [2]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 31.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]


In [3]:
import pandas as pd
import plotly.express as px
import os

# --- 1. ĐỌC DỮ LIỆU ---
# Đổi đường dẫn nếu file của bạn nằm ở chỗ khác
csv_path = "/home/xuanloc/DACN/ICIT/brrt_optimize/eval/output_optuna/optuna_history.csv"

if not os.path.exists(csv_path):
    print(f"Không tìm thấy file: {csv_path}")
    exit()

df = pd.read_csv(csv_path)

# Lọc bỏ các vòng bị lỗi (nếu có), chỉ lấy các vòng COMPLETE
if 'state' in df.columns:
    df = df[df['state'] == 'COMPLETE']

print(f"Đã load {len(df)} vòng thử nghiệm thành công.")

# --- 2. VẼ BIỂU ĐỒ 1: LỊCH SỬ TỐI ƯU (Optimization History) ---
# Trực quan hóa xem điểm Score giảm dần như thế nào qua thời gian
fig1 = px.scatter(df, x='number', y='value', trendline="lowess",
                  title='Lịch sử Tối ưu hóa (Điểm Score qua các vòng - Càng thấp càng tốt)',
                  labels={'number': 'Thứ tự Trial (Vòng lặp)', 'value': 'Điểm Score'})
fig1.write_html("optuna_history_plot.html")

# --- 3. VẼ BIỂU ĐỒ 2: PARALLEL COORDINATES ---
# Xem sự kết hợp giữa 4 tham số tạo ra điểm số tốt nhất (màu xanh/tối)
# Lấy tên các cột tham số (thường có tiền tố 'params_')
param_cols = [col for col in df.columns if col.startswith('params_')]

fig2 = px.parallel_coordinates(df, color="value",
                               dimensions=param_cols,
                               title='Phân tích đa biến (Parallel Coordinates)',
                               color_continuous_scale=px.colors.diverging.Tealrose)
fig2.write_html("optuna_parallel_plot.html")

print("\n✅ Đã tạo xong 2 biểu đồ phân tích tương tác!")
print("Hãy nhấp đúp vào file 'optuna_history_plot.html' và 'optuna_parallel_plot.html' để mở bằng trình duyệt Web.")

Đã load 50 vòng thử nghiệm thành công.

✅ Đã tạo xong 2 biểu đồ phân tích tương tác!
Hãy nhấp đúp vào file 'optuna_history_plot.html' và 'optuna_parallel_plot.html' để mở bằng trình duyệt Web.


In [4]:
import pandas as pd
import os

# --- 1. ĐƯỜNG DẪN FILE CSV ---
csv_path = "/home/xuanloc/DACN/ICIT/brrt_optimize/eval/output_optuna/optuna_history.csv"

if not os.path.exists(csv_path):
    print(f"Không tìm thấy file tại: {csv_path}")
else:
    # --- 2. ĐỌC VÀ LỌC DỮ LIỆU ---
    df = pd.read_csv(csv_path)

    # Chỉ lấy các vòng chạy hoàn tất (bỏ qua các vòng bị lỗi/crash nếu có)
    if 'state' in df.columns:
        df_completed = df[df['state'] == 'COMPLETE']
    else:
        df_completed = df

    if df_completed.empty:
        print("Không có vòng thử nghiệm nào chạy thành công trong file CSV.")
    else:
        # --- 3. TÌM BỘ SỐ TỐI ƯU NHẤT (SCORE THẤP NHẤT) ---
        # Hàm idxmin() sẽ trả về index của dòng có giá trị 'value' nhỏ nhất
        best_trial = df_completed.loc[df_completed['value'].idxmin()]

        # --- 4. IN KẾT QUẢ ---
        print("\n" + "="*50)
        print("🏆 BỘ THÔNG SỐ TỐI ƯU NHẤT TỪ FILE CSV 🏆")
        print("="*50)
        print(f"Trial Number : {int(best_trial['number'])}")
        print(f"Best Score   : {best_trial['value']:.4f}\n")
        
        print("--- CÁC THAM SỐ CẦN ĐIỀN VÀO C++ ---")
        # Optuna lưu các tham số với tiền tố 'params_', ta sẽ lọc chúng ra
        for col in df_completed.columns:
            if col.startswith('params_'):
                param_name = col.replace('params_', '')
                param_value = best_trial[col]
                print(f"  + {param_name}: {param_value}")
        print("="*50 + "\n")


🏆 BỘ THÔNG SỐ TỐI ƯU NHẤT TỪ FILE CSV 🏆
Trial Number : 11
Best Score   : 0.6135

--- CÁC THAM SỐ CẦN ĐIỀN VÀO C++ ---
  + lidar_radius: 17.5
  + n_blocks: 32
  + steer_length: 0.5
  + weight_grade: 3.5



# Normal test

In [ ]:
import pandas as pd
import os

# --- 1. ĐƯỜNG DẪN FILE CSV ---
csv_path = "/home/xuanloc/DACN/ICIT/brrt_optimize/eval/output_optuna/optuna_history.csv"

if not os.path.exists(csv_path):
    print(f"Không tìm thấy file tại: {csv_path}")
else:
    # --- 2. ĐỌC VÀ LỌC DỮ LIỆU ---
    df = pd.read_csv(csv_path)

    # Chỉ lấy các vòng chạy hoàn tất (bỏ qua các vòng bị lỗi/crash nếu có)
    if 'state' in df.columns:
        df_completed = df[df['state'] == 'COMPLETE']
    else:
        df_completed = df

    if df_completed.empty:
        print("Không có vòng thử nghiệm nào chạy thành công trong file CSV.")
    else:
        # --- 3. TÌM BỘ SỐ TỐI ƯU NHẤT (SCORE THẤP NHẤT) ---
        # Hàm idxmin() sẽ trả về index của dòng có giá trị 'value' nhỏ nhất
        best_trial = df_completed.loc[df_completed['value'].idxmin()]

        # --- 4. IN KẾT QUẢ ---
        print("\n" + "="*50)
        print("🏆 BỘ THÔNG SỐ TỐI ƯU NHẤT TỪ FILE CSV 🏆")
        print("="*50)
        print(f"Trial Number : {int(best_trial['number'])}")
        print(f"Best Score   : {best_trial['value']:.4f}\n")
        
        print("--- CÁC THAM SỐ CẦN ĐIỀN VÀO C++ ---")
        # Optuna lưu các tham số với tiền tố 'params_', ta sẽ lọc chúng ra
        for col in df_completed.columns:
            if col.startswith('params_'):
                param_name = col.replace('params_', '')
                param_value = best_trial[col]
                print(f"  + {param_name}: {param_value}")
        print("="*50 + "\n")

KeyError: 'value'